In [1]:
import numpy as np
import scipy.linalg as la
from data import generate_data, init_geodesic
from solver import GeodesicEstimator
from utils import projection_mse, compute_PSNR, compute_SSIM, normalize_np
from functools import partial
from data_obs import generate_mc

## Importing Data

In [2]:
with np.load('img_utils/mouse_data.npz') as data:
    data = data['arr_0']

In [3]:
x, y, frames = data.shape
print('Total Number of Frames:', frames)

Total Number of Frames: 80


In [5]:
vec_data = np.reshape(data, (-1, frames), order='F')
vec_data = vec_data / 255
vec_data_norm = vec_data - np.mean(vec_data, axis=0) # de-meaning data

In [6]:
Xs = []

ell = 5 # Number of data points per time index

st = 0
en = ell

for i in range(frames // ell):
    Xs.append(vec_data_norm[:, st:en])
    
    st += ell
    en += ell
    
T = len(Xs)

## Training

In [7]:
def spectral_init(Ys, frac_observed):
    """
    Performing spectral initialization for missing data case.
    """

    T = len(Ys)
    ell = Ys[0].shape[1]
    
    Xs_init = []

    for t in range(T):
        temp = []
        for i in range(ell):
            spect_mat = np.expand_dims( (1 / frac_observed)*Ys[t][:, i], axis=1 )
            temp.append(np.expand_dims( np.linalg.svd( spect_mat )[0][:, 0], axis=1 ))

        Xs_init.append(np.concatenate(temp, axis=1))
        
    return Xs_init

In [8]:
FRAC_OBSERVED = 0.5

NTRIALS = 1
NITERS = 50
RANK = 6

# For AltMin
TOL = 1e-2
N_INNER_ITERS = 10

# For Geodesic
LAMBDA_VAL = 0.01

### Generating Data

In [ ]:
err_data_mm = np.zeros((NTRIALS, NITERS))
err_obj_func = np.zeros((NTRIALS, NITERS))


for trial in range( NTRIALS ):
    print('Trial Number: ', trial)

    # Generating data
    forward, transpose, masks = generate_mc(d=Xs[0].shape[0], ell=ell, T=T,frac_observed=FRAC_OBSERVED)
    
    # Generate masked data
    Ys = []

    for i, Xi in enumerate(Xs):
        Ys.append( forward(Xi, masks[i]) + np.random.randn(*Xi.shape)*SIGMA )

    def callback(itr, geo, data):
        for i in range(T):
            err_data_mm[trial, itr] += np.linalg.norm( Xs[i]-data["Xs"][i] )**2 / np.linalg.norm(Xs[i])**2
        err_data_mm[trial, itr] = err_data_mm[trial, itr] / T
        err_obj_func[trial, itr] =  obj_func(Ys=Ys, Xs=data["Xs"], forward=forward, mask=np.concatenate(masks, axis=1))
        print('Iteration:', itr, 'MSE:', err_data_mm[trial, itr], "OBJ FUNC:", err_obj_func[trial, itr])

        return None
    
    # Initialization via zero-filled reconstruction
    Xs_init = []
    for i in range(T):
        Xs_init.append( np.real(transpose(Ys[i], None)) )

    # Initialization via data sharing
    shared_Y = kspace_data_share(Ys=Ys, masks=masks, dims=(x,y))
    shared_Ys = break_data(data=shared_Y, ell=ell)

    Xs_init = []
    for i in range(T):
         Xs_init.append( np.real(transpose(shared_Ys[i], None)) )

    # Compute initial MSE
    val = 0
    for i in range(T):
        val += np.linalg.norm( Xs[i]-Xs_init[i] )**2 / np.linalg.norm(Xs[i])**2
    print("Initial MSE:", val/T)

    geo = init_geodesic(d=int(x*y), k=RANK, Xs=Xs_init, init="pca") # Initialize geodesic
    ts = np.linspace(0.0, 1.0, T, endpoint=True) # Timestamps

    data_init = {
    "Xs":            Xs_init,
    "ts":            ts,
    "measurements":  Ys,
    "forward":       forward,
    "transpose":     transpose,
    "masks":         masks,
    }

    estimator = GeodesicEstimator(rank=RANK)
    geo_mm_est = estimator.fit(
                    geo=geo,
                    data=data_init,
                    callback=callback,
                    niters=NITERS,
                    L=1
    )